# Transformer Circuits: A Mathematical Framework

Transformers aren't a mystery -- they're a collection of components that read from and write to a shared "residual stream." Once you see it this way, you can reverse-engineer what each piece does. That's what this notebook is about.

We'll load GPT-2 small with [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens) and take it apart. You should already be comfortable with attention, MLPs, residual connections, linear algebra, and PyTorch.

---
## Section 1: The Residual Stream View

Here's the key reframing from [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html): don't think of a transformer as a sequence of layers. Think of it as a **residual stream** -- a sequence of vectors (one per token position) that persists across the entire forward pass.

Each attention head and MLP layer **reads from** the residual stream (via its input projection) and **writes to** the residual stream (via its output, which gets added back in). The residual stream is the "communication channel" between all components.

### Key equation

Let $x_0$ be the initial token embeddings (+ positional embeddings). After $l$ layers, the residual stream is:

$$x_l = x_0 + \sum_{i < l} \left( \text{attn}_i(x) + \text{mlp}_i(x) \right)$$

Each component's contribution is additive. This gives us three powerful properties:
- You can **analyze components independently** (their outputs simply sum).
- The output logits are a **linear function** of every component's output (up to the final LayerNorm).
- You can ask: "How much does head L2H7 contribute to the logit for token X?" and get a precise answer.

This additive structure is what makes mechanistic interpretability tractable.

In [ ]:
import torch
import transformer_lens
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt
import numpy as np

# Load GPT-2 small
model = HookedTransformer.from_pretrained("gpt2-small")
print(f"Model: {model.cfg.model_name}")
print(f"Layers: {model.cfg.n_layers}, Heads: {model.cfg.n_heads}, d_model: {model.cfg.d_model}")
print(f"d_head: {model.cfg.d_head}, d_mlp: {model.cfg.d_mlp}")

In [ ]:
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Inspect residual stream at each layer
for layer in range(model.cfg.n_layers):
    resid = cache[f"blocks.{layer}.hook_resid_post"]
    print(f"Layer {layer:2d}: residual stream shape = {resid.shape}, norm = {resid.norm(dim=-1).mean():.2f}")

Notice how the residual stream norm **grows** across layers. Each attention head and MLP writes its output into the stream, so the magnitude accumulates. This is a basic but important sanity check: the residual stream is not static; it is progressively enriched by each component.

---
### Why Attention Decomposes into Independent Head Contributions

The additivity of the residual stream would be useless if multi-head attention entangled heads at the output. Fortunately, it doesn't -- and this is **the** structural property that makes circuits analysis possible.

**Starting point.** The standard multi-head attention equation is:

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)\, W_O$$

where each head computes $\text{head}_h = A_h\, X\, W_V^h$ with attention weights $A_h = \mathrm{softmax}\!\left(\frac{X W_Q^h (W_K^h)^\top X^\top}{\sqrt{d_k}}\right)$.

**Block structure of $W_O$.** The concatenation-then-project operation is equivalent to a partitioned view: $W_O$ can be written in block form $W_O = \begin{bmatrix} W_O^1 \\ W_O^2 \\ \vdots \\ W_O^H \end{bmatrix}$ where each $W_O^h \in \mathbb{R}^{d_{\text{head}} \times d_{\text{model}}}$. Each block operates only on the corresponding head's output.

**So we get:**

$$\text{MultiHead}(X) = \sum_{h=1}^{H} \text{head}_h \, W_O^h = \sum_{h=1}^{H} A_h \, X \, W_V^h \, W_O^h$$

Each head's contribution to the residual stream is:

$$\Delta x^{(h)} = A_h \, X \, W_V^h \, W_O^h$$

where $A_h = \mathrm{softmax}\!\left(\frac{X W_Q^h {(W_K^h)}^\top X^\top}{\sqrt{d_k}}\right) \in \mathbb{R}^{T \times T}$ and $W_V^h W_O^h \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$ is exactly the OV circuit.

**Why this matters.** The residual stream after attention is $x' = x + \sum_h \Delta x^{(h)}$, a sum of $H$ independent terms plus the skip connection. Each head's contribution can be:
1. **Computed independently** -- no head needs another head's output within the same layer.
2. **Ablated independently** -- setting one head's output to zero doesn't affect other heads.
3. **Analyzed independently** -- the QK and OV circuits of each head tell the full story of what that head does.

**Important caveat: MLPs are NOT decomposable.** The MLP sublayer applies a pointwise nonlinearity: $\text{MLP}(x) = W_{\text{out}} \cdot \sigma(W_{\text{in}} \, x + b_{\text{in}}) + b_{\text{out}}$. Because $\sigma$ (GELU/ReLU) is applied element-wise *after* a single linear projection, there's no natural decomposition of the MLP output into independent additive terms. This asymmetry -- attention is linearly decomposable, MLPs are not -- is why current circuits work focuses heavily on attention heads. Understanding MLP computation remains a major open problem.

---
## Section 2: Attention Head Decomposition -- QK and OV Circuits

Every attention head does two functionally distinct things, and you can factor them into two separate circuits:

### QK Circuit ($W_Q^T W_K$): "Where to attend"

The QK circuit determines the **attention pattern** -- which source positions each destination position attends to:

$$A = \text{softmax}\left(\frac{x W_Q W_K^T x^T}{\sqrt{d_{\text{head}}}}\right)$$

The bilinear form $W_Q W_K^T$ (a $d_{\text{model}} \times d_{\text{model}}$ matrix) tells you: given a query vector and a key vector from the residual stream, how strongly should this head attend?

### OV Circuit ($W_V W_O$): "What to write"

The OV circuit determines **what information** gets moved from attended-to positions into the residual stream:

$$\text{attn\_out} = A \cdot x W_V W_O$$

The matrix $W_V W_O$ (also $d_{\text{model}} \times d_{\text{model}}$) defines the linear transformation applied to each source token's residual stream vector before writing the result back.

### Why this decomposition matters

You can analyze these two circuits **independently**:
- The QK circuit controls **information routing** (the attention pattern).
- The OV circuit controls **information transformation** (what gets written).

A head might have a simple QK pattern (e.g., "attend to previous token") but a complex OV transformation, or vice versa.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

layer, head = 0, 0

W_Q = model.W_Q[layer, head]  # (d_model, d_head)
W_K = model.W_K[layer, head]  # (d_model, d_head)
W_V = model.W_V[layer, head]  # (d_model, d_head)
W_O = model.W_O[layer, head]  # (d_head, d_model)

# QK circuit: what the head attends to
QK = W_Q @ W_K.T  # (d_model, d_model)
print(f"QK circuit shape: {QK.shape}")

# OV circuit: what the head writes
OV = W_V @ W_O  # (d_model, d_model)
print(f"OV circuit shape: {OV.shape}")

# Visualize eigenvalue spectra with plotly
qk_eigs = torch.linalg.eigvalsh(QK.float().cpu()).detach().numpy()
qk_eigs_sorted = sorted(qk_eigs, reverse=True)[:50]

ov_svd = torch.linalg.svdvals(OV.float().cpu()).detach().numpy()[:50]

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"QK Circuit Eigenvalues (L{layer}H{head})",
    f"OV Circuit Singular Values (L{layer}H{head})"
])

fig.add_trace(
    go.Scatter(
        x=list(range(len(qk_eigs_sorted))),
        y=qk_eigs_sorted,
        mode="lines+markers",
        marker=dict(size=4),
        hovertemplate="Index: %{x}<br>Eigenvalue: %{y:.4f}<extra></extra>",
        name="QK Eigenvalues",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=list(range(len(ov_svd))),
        y=ov_svd.tolist(),
        mode="lines+markers",
        marker=dict(size=4),
        hovertemplate="Index: %{x}<br>Singular Value: %{y:.4f}<extra></extra>",
        name="OV Singular Values",
    ),
    row=1, col=2,
)

fig.update_xaxes(title_text="Index", row=1, col=1)
fig.update_xaxes(title_text="Index", row=1, col=2)
fig.update_yaxes(title_text="Eigenvalue", row=1, col=1)
fig.update_yaxes(title_text="Singular Value", row=1, col=2)

fig.update_layout(height=400, width=900, showlegend=False)
fig.show()

The eigenvalue spectrum of the QK circuit reveals the **effective rank** of the attention pattern computation. A few large eigenvalues indicate the head attends based on a low-dimensional subspace of the residual stream. Similarly, the singular value spectrum of the OV circuit shows how many independent directions the head uses when writing back to the stream.

---
## Section 3: Attention Patterns

Let's visualize what attention heads actually attend to on real input. Different heads learn qualitatively different patterns:

- **Previous token heads**: attend primarily to the immediately preceding token
- **Induction heads**: attend to the token that followed a previous occurrence of the current token
- **Duplicate token heads**: attend to previous occurrences of the same token
- **Positional heads**: attend based on relative or absolute position rather than content

We'll use a sentence with repeated names to make some of these patterns pop out.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

prompt = "When Mary and John went to the store, John gave a drink to"
logits, cache = model.run_with_cache(prompt)
tokens = [model.tokenizer.decode(t) for t in model.to_tokens(prompt)[0]]

# Plot attention patterns for first 3 layers, 4 heads each using plotly subplots
fig = make_subplots(
    rows=3, cols=4,
    subplot_titles=[f"L{layer}H{head}" for layer in range(3) for head in range(4)],
    horizontal_spacing=0.05,
    vertical_spacing=0.08,
)

for layer in range(3):
    for head in range(4):
        attn_pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().cpu().numpy()
        heatmap = go.Heatmap(
            z=attn_pattern,
            x=tokens,
            y=tokens,
            colorscale="Blues",
            showscale=(layer == 0 and head == 3),  # show colorbar only once
            hovertemplate="Query: %{y}<br>Key: %{x}<br>Weight: %{z:.4f}<extra></extra>",
        )
        fig.add_trace(heatmap, row=layer + 1, col=head + 1)

fig.update_layout(
    title_text="Attention Patterns (first 3 layers, 4 heads each)",
    height=900,
    width=1100,
)
fig.show()

Look for:
- A strong **diagonal** (one below the main diagonal) indicating a previous-token head.
- Columns that light up on specific tokens (e.g., "John") indicating content-based attention.
- Uniform rows indicating a head that spreads attention broadly (often a "background" or positional head).

---
## Section 4: Composition — How Heads Talk to Each Other

Because all components read from and write to the same residual stream, heads in **different layers** can compose. A head in layer 1 can use the output of a head in layer 0 as part of its input.

There are three types of composition, depending on which input pathway the downstream head uses:

### Q-composition
Head B (layer $l_2$) uses head A's output (layer $l_1 < l_2$) as part of its **query** input. This means head A's output helps determine *where* head B attends.

### K-composition
Head B uses head A's output as part of its **key** input. Head A's output helps determine *what head B matches against*.

### V-composition
Head B uses head A's output as part of its **value** input. Head A's output is directly transformed and passed through head B.

### Measuring composition strength

The composition strength between head A (output: $W_V^A W_O^A$) and head B (query: $W_Q^B$) for Q-composition is:

$$\text{Q-comp}(A \to B) = \| W_V^A W_O^A W_Q^B \|_F$$

### Induction heads as a composition example

The canonical example is **induction heads**: a previous-token head in layer 0 composes with an induction head in layer 1 via **K-composition** to implement the rule:

> "If I've seen token A followed by token B earlier, and I now see A again, predict B."

The previous-token head writes "the identity of the previous token" into the residual stream. The induction head uses this as its key, so it attends to positions where the previous token matches the current token — and then copies the next token.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Q-composition: How much does L1 head use L0 head output as query input?
n_heads = model.cfg.n_heads
q_comp_scores = torch.zeros(n_heads, n_heads)

for h0 in range(n_heads):
    W_OV_0 = model.W_V[0, h0] @ model.W_O[0, h0]  # L0 head output
    for h1 in range(n_heads):
        W_Q_1 = model.W_Q[1, h1]  # L1 head query
        # Composition: L0 output -> L1 query
        composed = W_OV_0 @ W_Q_1
        q_comp_scores[h0, h1] = composed.norm().item()

head_labels = [f"H{i}" for i in range(n_heads)]

fig = px.imshow(
    q_comp_scores.detach().numpy(),
    x=head_labels,
    y=head_labels,
    labels=dict(x="L1 Head (query)", y="L0 Head (output)", color="Frobenius norm"),
    color_continuous_scale="Reds",
    title="Q-Composition Scores: L0 -> L1",
)
fig.update_traces(
    hovertemplate="L0 Head: %{y}<br>L1 Head: %{x}<br>Frobenius norm: %{z:.4f}<extra></extra>"
)
fig.update_layout(height=600, width=700)
fig.show()

Bright spots in this heatmap indicate pairs of heads with strong Q-composition — meaning the layer-0 head's output significantly influences where the layer-1 head attends. You can compute analogous K-composition and V-composition matrices by replacing $W_Q^B$ with $W_K^B$ or examining the value path.

### The Full Composition Formula

Let us make the composition mechanics precise. Consider head $A$ in layer $l$ with OV circuit $W_{OV}^A = W_V^A W_O^A$ and head $B$ in layer $l' > l$.

**V-composition** (head $A$'s output flows through head $B$'s value pathway):

The composed circuit matrix is $W_{OV}^A \cdot W_{OV}^B \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$. This matrix describes the end-to-end linear map: "head $A$ reads from the residual stream, transforms via its OV circuit, writes to the residual stream, and head $B$ reads this, transforms via *its* OV circuit, and writes the result." In effect, $W_{OV}^A \cdot W_{OV}^B$ is a rank-$d_{\text{head}}$ bottlenecked composition of two linear maps.

**K-composition** (head $A$'s output is used as key input by head $B$):

The composed circuit is $W_{OV}^A \cdot W_{QK}^B = W_{OV}^A \cdot W_Q^B (W_K^B)^\top$. More precisely, the attention score contribution from the $A \to B$ K-composition path is:

$$\alpha_{ij}^{A \to B} \propto x_i^\top\, W_Q^B\, (W_K^B)^\top\, (W_{OV}^A)^\top\, x_j$$

This measures: "does the content that head $A$ writes into position $j$'s residual stream cause head $B$ at position $i$ to attend to position $j$?"

**Q-composition** (head $A$'s output is used as query input by head $B$):

Analogously, $W_{OV}^A$ appears in the query pathway:

$$\alpha_{ij}^{A \to B} \propto x_i^\top\, (W_{OV}^A)^\top\, W_Q^B\, (W_K^B)^\top\, x_j$$

**Measuring composition strength.** The Frobenius norm $\|W_{OV}^A \cdot W_{QK}^B\|_F$ gives a scalar measure of how strongly head $A$ composes with head $B$ via K-composition. The intuition: if the image of $W_{OV}^A$ is nearly orthogonal to the row space of $W_{QK}^B$, the composed matrix has small norm and the heads do not meaningfully interact. If they overlap substantially, the norm is large and the composition is strong.

**Important subtlety:** These formulas describe the *direct* composition path. In reality, head $B$ reads from the full residual stream $x_0 + \sum_{h} \Delta x^{(h)}$, so the A-to-B composition signal competes with all other residual stream content. The Frobenius norm measures the *capacity* for composition, not the *actual* composition on any given input (which depends on the data distribution).

---
## Section 5: Induction Heads — A Complete Circuit

Induction heads are the simplest known **multi-layer circuit** in transformers. They implement a form of in-context learning:

> If the model has seen the bigram $[A][B]$ earlier in context, and token $[A]$ appears again, predict $[B]$.

### The two-head mechanism

1. **Previous token head** (layer 0): Attends from position $i$ to position $i-1$. Its OV circuit copies information about "the previous token" into the residual stream at each position.

2. **Induction head** (layer 1): Uses **K-composition** with the previous token head. Its key input now contains information about "the token before me". So when it computes attention, it matches positions where "the token before me" equals "the current query token" — and then copies the token at that position (which is the $[B]$ that followed $[A]$).

### Detection strategy

We can detect induction heads by feeding the model a **repeated random sequence** and checking which heads attend to the "induction offset" — the position that is exactly one sequence-length back plus one.

---
## Exercises

### Exercise 1: Find the Previous Token Heads

Write code to identify which attention heads in GPT-2 most strongly attend to the previous token position. For each head across all layers, compute the average attention weight paid to position `(t-1)` across all destination positions `t` on a batch of text. Rank heads by this score and report the top 5.

**Hint**: For a head's attention pattern of shape `(seq, seq)`, the "previous token" attention at destination `t` is `attn[t, t-1]`. Average this over all valid `t`.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Create a repeated sequence to trigger induction heads
repeated_tokens = model.to_tokens("ABCDEFGHIJ " * 3)  # Repeat to create induction signal

logits, cache = model.run_with_cache(repeated_tokens)

# Score each head for induction behavior:
# An induction head attends to the token after the previous occurrence of the current token
# This creates a diagonal stripe offset by the sequence length
induction_scores = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

for layer in range(model.cfg.n_layers):
    attn = cache[f"blocks.{layer}.attn.hook_pattern"][0]  # (n_heads, seq, seq)
    seq_len = attn.shape[-1]
    for head in range(model.cfg.n_heads):
        # Simple induction score: average attention paid to offset diagonal
        score = 0
        count = 0
        for i in range(11, seq_len):  # Start after first repeat
            # The induction target is the token that followed the previous occurrence
            target = i - 10  # Offset by repeat length (approximate)
            if 0 <= target < seq_len:
                score += attn[head, i, target].item()
                count += 1
        induction_scores[layer, head] = score / max(count, 1)

layer_labels = [f"L{i}" for i in range(model.cfg.n_layers)]
head_labels = [f"H{i}" for i in range(model.cfg.n_heads)]

fig = px.imshow(
    induction_scores.detach().numpy(),
    x=head_labels,
    y=layer_labels,
    labels=dict(x="Head", y="Layer", color="Induction Score"),
    color_continuous_scale="Reds",
    aspect="auto",
    title="Induction Head Detection (higher = more induction-like)",
)
fig.update_traces(
    hovertemplate="Layer: %{y}<br>Head: %{x}<br>Induction Score: %{z:.4f}<extra></extra>"
)
fig.update_layout(height=500, width=900)
fig.show()

# Print top induction heads
flat_idx = induction_scores.flatten().argsort(descending=True)[:5]
print("Top 5 induction-like heads:")
for idx in flat_idx:
    l = idx.item() // model.cfg.n_heads
    h = idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: score = {induction_scores[l, h]:.4f}")

You should see induction heads clustering in the **early-to-mid layers** (typically layers 1-5 in GPT-2 small). These heads drive a huge fraction of the model's in-context learning ability. [Olsson et al. (2022)](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) showed that induction heads emerge during a phase change in training and are causally responsible for in-context learning improvements.

---
### Running Example: IOI — Attention Patterns

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

Here we apply this notebook's technique — examining attention patterns — to identify heads that attend from the final token position back to "Mary". These are candidates for **name mover heads**, which copy the indirect object's name to the output.

In [ ]:
# Running Example: IOI — Name Mover Head Detection via Attention Patterns
prompt = "When Mary and John went to the store, John gave a drink to"
logits, cache = model.run_with_cache(prompt)
tokens = [model.tokenizer.decode(t) for t in model.to_tokens(prompt)[0]]

print(f"Prompt: '{prompt}'")
print(f"Tokens: {tokens}")
print(f"\nScanning layers 7-9 for heads that attend from the final position back to ' Mary':")
print(f"(These are candidate name mover heads)\n")

# Show attention from last position to all others, for layers 7-9
for layer in [7, 8, 9]:
    attn = cache[f"blocks.{layer}.attn.hook_pattern"][0]  # (n_heads, seq, seq)
    for head in range(model.cfg.n_heads):
        attn_to_mary = attn[head, -1, 1].item()  # attention from last pos to "Mary" (token index 1)
        if attn_to_mary > 0.1:
            print(f"L{layer}H{head}: {attn_to_mary:.3f} attention to ' Mary'")

---
## Exercises

### Exercise 1: Decompose an Attention Head

Pick a specific head in GPT-2 small (e.g., Layer 0, Head 7). Extract its W_Q, W_K, W_V, W_O matrices. Compute the full QK circuit (W_E^T @ W_Q @ W_K^T @ W_E) and OV circuit (W_E @ W_V @ W_O @ W_U). Analyze:
- What tokens does this head attend to (via the QK circuit)?
- What does it copy/write (via the OV circuit)?

Find the top 10 (query_token, key_token) pairs by QK score and the top 10 (input_token, output_token) pairs by OV score.

<details>
<summary>Hint</summary>

Use `model.W_Q[layer, head]`, `model.W_K[layer, head]`, etc. to extract per-head weight matrices. The embedding matrix is `model.W_E` and the unembedding matrix is `model.W_U`. The full QK circuit has shape `(d_vocab, d_vocab)` — you may want to work with a subset of the vocabulary to keep computation manageable. Use `model.to_single_token()` to convert tokens for inspection.

</details>

In [ ]:
# Exercise 1: Decompose an Attention Head
# Choose layer 0, head 7 — TODO: Try modifying this to explore other heads!
layer, head = 0, 7

# Extract per-head weight matrices
W_Q = model.W_Q[layer, head]  # (d_model, d_head)
W_K = model.W_K[layer, head]  # (d_model, d_head)
W_V = model.W_V[layer, head]  # (d_model, d_head)
W_O = model.W_O[layer, head]  # (d_head, d_model)

# Get embedding and unembedding matrices
W_E = model.W_E  # (d_vocab, d_model)
W_U = model.W_U  # (d_model, d_vocab)

# 1. Compute the full QK circuit: W_E @ W_Q @ W_K^T @ W_E^T
#    This has shape (d_vocab, d_vocab) -- use a subset of vocab to keep it manageable
#    TODO: Try changing the vocab subset size!
vocab_subset = 1000
full_QK = W_E[:vocab_subset] @ W_Q @ W_K.T @ W_E[:vocab_subset].T

# 2. Compute the full OV circuit: W_E @ W_V @ W_O @ W_U
#    This has shape (d_vocab, d_vocab) -- again use a subset
full_OV = W_E[:vocab_subset] @ W_V @ W_O @ W_U[:, :vocab_subset]

# 3. Find the top 10 (query_token, key_token) pairs by QK score
qk_flat = full_QK.detach().cpu().flatten()
top_qk_indices = torch.topk(qk_flat, 10).indices
print(f"Top 10 (query, key) pairs by QK score for L{layer}H{head}:")
for idx in top_qk_indices:
    row = idx.item() // vocab_subset
    col = idx.item() % vocab_subset
    q_tok = model.tokenizer.decode([row])
    k_tok = model.tokenizer.decode([col])
    score = full_QK[row, col].item()
    print(f"  query='{q_tok}', key='{k_tok}', score={score:.3f}")

# 4. Find the top 10 (input_token, output_token) pairs by OV score
print(f"\nTop 10 (input, output) pairs by OV score for L{layer}H{head}:")
ov_flat = full_OV.detach().cpu().flatten()
top_ov_indices = torch.topk(ov_flat, 10).indices
for idx in top_ov_indices:
    row = idx.item() // vocab_subset
    col = idx.item() % vocab_subset
    in_tok = model.tokenizer.decode([row])
    out_tok = model.tokenizer.decode([col])
    score = full_OV[row, col].item()
    print(f"  input='{in_tok}', output='{out_tok}', score={score:.3f}")

### Exercise 2: Verify the Residual Stream View

Run GPT-2 on a prompt. Cache all activations. Manually sum the contributions (embed + pos_embed + all attention outputs + all MLP outputs up to layer L) and verify that the sum equals the residual stream at layer L.

This exercise verifies the core claim of the residual stream framework: the residual stream at any layer is the sum of all prior component outputs.

<details>
<summary>Hint</summary>

Use `model.run_with_cache(prompt)` to get all activations. The key tensors are: `cache['embed']` (token embeddings), `cache['pos_embed']` (positional embeddings), `cache['attn_out', layer]` (attention output at each layer), and `cache['mlp_out', layer]` (MLP output at each layer). The residual stream after layer L is `cache['resid_post', L]`. Sum embed + pos_embed + all attn_out and mlp_out for layers 0..L and compare using `torch.allclose()`.

</details>

In [ ]:
# Exercise 2: Verify the Residual Stream View
prompt = "The capital of France is"
logits, cache = model.run_with_cache(prompt)

# 1. Get the initial embeddings
embed = cache['embed']           # token embeddings
pos_embed = cache['pos_embed']   # positional embeddings

# 2. Sum all contributions up to layer L
# TODO: Try changing L to verify at different depths!
L = 5
manual_sum = embed + pos_embed
for layer in range(L + 1):
    manual_sum = manual_sum + cache['attn_out', layer] + cache['mlp_out', layer]

# 3. Get the actual residual stream at layer L
actual_resid = cache['resid_post', L]

# 4. Compare: are they equal (up to floating point)?
print(f"Manual sum shape: {manual_sum.shape}")
print(f"Actual resid shape: {actual_resid.shape}")
print(f"Are they close? {torch.allclose(manual_sum, actual_resid, atol=1e-4)}")
print(f"Max absolute difference: {(manual_sum - actual_resid).abs().max().item():.6e}")

---
## Section 6: Key Takeaways & Further Reading

### What you should take away from this notebook

1. **Residual stream as communication channel**: A transformer is best understood as a residual stream that components read from and write to additively. This makes individual contributions separable and analyzable.

2. **QK/OV decomposition**: Each attention head has two independent circuits -- QK for routing (where to attend) and OV for transformation (what to write). You can study these via their eigenvalue/singular value spectra.

3. **Cross-layer composition**: Heads in different layers compose via Q-, K-, and V-composition, enabling multi-step algorithms. You can measure composition strength via Frobenius norms of composed weight matrices.

4. **Induction heads**: The canonical example of a cross-layer circuit. A previous-token head (L0) K-composes with an induction head (L1) to implement "if A B ... A, predict B". This is a complete, mechanistically understood algorithm inside the transformer.

### Further reading

- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) (Elhage et al., 2021) -- the foundational paper for this entire framework
- [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) (Olsson et al., 2022) -- deep dive on induction heads and their role in in-context learning
- [TransformerLens documentation and tutorials](https://neelnanda-io.github.io/TransformerLens/) -- Neel Nanda's library and guides
- [200 Concrete Open Problems in Mechanistic Interpretability](https://www.alignmentforum.org/s/yivyHaCAmMJ3CqSyj) -- Neel Nanda's problem list for getting started

### Next

**Notebook 02 -- Superposition**: Why don't features correspond to individual neurons? We'll explore the superposition hypothesis and why networks represent more features than they have dimensions.